In [1]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

Workspace name: quick-starts-ws-305885
Azure region: eastus2
Subscription id: cdbe0b43-92a0-4715-838a-f2648cc7ad21
Resource group: aml-quickstarts-305885


In [2]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = "cpu-cluster"

# TODO: Create compute cluster
# Use vm_size = "Standard_D2_V2" in your provisioning configuration.
# max_nodes should be no greater than 4.

### YOUR CODE HERE ###
try:
    compute_target = ComputeTarget(workspace=ws, name=cluster_name)
    print("Found existing cluster, using it")
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(vm_size="Standard_D2_V2",
                                                           max_nodes=4)
    compute_target = ComputeTarget.create(ws, cluster_name, compute_config)

compute_target.wait_for_completion(show_output=True)

InProgress..
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


In [3]:
from azureml.widgets import RunDetails
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os

# Specify parameter sampler
# ps = ### YOUR CODE HERE ###
ps = RandomParameterSampling({
    "--C": uniform(0.01, 10),
    "--max_iter": choice(50, 100, 200, 300)
})

# Specify a Policy
# policy = ### YOUR CODE HERE ###
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1)

if "training" not in os.listdir():
    os.mkdir("./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(name='sklearn-env', file_path='conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
# src = ### YOUR CODE HERE ###
src = ScriptRunConfig(source_directory=".",
                      script="train.py",
                      compute_target=compute_target,
                      environment=sklearn_env)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
# hyperdrive_config = ### YOUR CODE HERE ###
hyperdrive_config = HyperDriveConfig(run_config=src,
                                     hyperparameter_sampling=ps,
                                     policy=policy,
                                     primary_metric_name="Accuracy",
                                     primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
                                     max_total_runs=20,
                                     max_concurrent_runs=4)

Now the notebook can’t read the URL either:

In [5]:
from azureml.data.dataset_factory import TabularDatasetFactory
import os

url = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
ds_local = TabularDatasetFactory.from_delimited_files(path=url)
ds_local.to_pandas_dataframe().to_csv("bankmarketing_train.csv", index=False)

print(os.listdir())

Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


DatasetValidationError: DatasetValidationError:
	Message: Failed to validate the data.If data is inaccessible, please set validate to False.
Authentication failed when trying to access the stream. Make sure you have correct permissions set up. For troubleshooting help, see: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-workspace-diagnostic-api?view=azureml-api-2| session_id=901ba199-c3b7-472f-8968-bae650e9e06e
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "Failed to validate the data.If data is inaccessible, please set validate to False.\nAuthentication failed when trying to access the stream. Make sure you have correct permissions set up. For troubleshooting help, see: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-workspace-diagnostic-api?view=azureml-api-2| session_id=901ba199-c3b7-472f-8968-bae650e9e06e"
    }
}

so we save csv file on the left side 

In [8]:
import urllib.request, zipfile, io, os
import pandas as pd

zip_url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(zip_url).read()))
inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
df = pd.read_csv(inner.open("bank-additional/bank-additional-full.csv"), sep=";")

df.to_csv("bankmarketing_train.csv", index=False)
print(df.shape)
print(os.listdir())

print(os.getcwd())    # the folder Python is working in


(41188, 21)
['.amlignore', '.amlignore.amltmp', '.ipynb_aml_checkpoints', 'bankmarketing_train.csv', 'conda_dependencies.yml', 'outputs', 'README.md', 'train.py', 'train.py.amltmp', 'training', 'udacity-project', 'udacity-project.ipynb', 'udacity-project.ipynb.amltmp']
/mnt/batch/tasks/shared/LS_root/mounts/clusters/computeshiva/code/Users/odl_user_305885


In [9]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

### YOUR CODE HERE ###
hyperdrive_run = exp.submit(config=hyperdrive_config)
RunDetails(hyperdrive_run).show()
hyperdrive_run.wait_for_completion(show_output=True)

_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

RunId: HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b
Web View: https://ml.azure.com/runs/HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b?wsid=/subscriptions/cdbe0b43-92a0-4715-838a-f2648cc7ad21/resourcegroups/aml-quickstarts-305885/workspaces/quick-starts-ws-305885&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-11-07T10:21:45.5616840Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-11-07T10:21:46.1072466Z][SCHEDULER][INFO]Scheduling job, id='HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_0' 
[2026-11-07T10:21:46.1736998Z][SCHEDULER][INFO]Scheduling job, id='HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_2' 
[2026-11-07T10:21:46.1746488Z][SCHEDULER][INFO]Scheduling job, id='HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_1' 
[2026-11-07T10:21:46.2145601Z][SCHEDULER][INFO]Scheduling job, id='HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_3' 
[2026-11-07T10:21:46.9361183Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_1' 
[2026-11-07T10:2

{'runId': 'HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b',
 'target': 'cpu-cluster',
 'status': 'Completed',
 'startTimeUtc': '2026-11-07T10:21:43.388263Z',
 'endTimeUtc': '2026-11-07T10:27:49.792978Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': '260ee190-21ed-4570-a882-9ff93c9ea0c0',
  'user_agent': 'python/3.10.0 (Linux-6.8.0-1070-azure-x86_64-with-glibc2.35) msrest/0.7.1 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.61.0',
  'best_child_run_id': 'HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_5',
  'score': '0.9115076474872542',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_5'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryValues': {'amlClientType': 'azureml-sdk-tra

In [10]:
import joblib
# Get your best run and save the model from that run.

### YOUR CODE HERE ###
import os

best_run = hyperdrive_run.get_best_run_by_primary_metric()
best_metrics = best_run.get_metrics()

print("Best run ID:", best_run.id)
print("Accuracy:", best_metrics["Accuracy"])
print("Parameters:", best_run.get_details()["runDefinition"]["arguments"])

# Register the model in the workspace
model = best_run.register_model(model_name="hyperdrive_best_model",
                                model_path="outputs/model.joblib")
print("Registered model:", model.name, "version", model.version)

# Download the model file and save it locally
os.makedirs("outputs", exist_ok=True)
best_run.download_file(name="outputs/model.joblib",
                       output_file_path="outputs/hyperdrive_best_model.joblib")
best_model = joblib.load("outputs/hyperdrive_best_model.joblib")
print(best_model)

Best run ID: HD_a89745b6-6dea-4c20-b044-7deeb51c3e6b_5
Accuracy: 0.9115076474872542
Parameters: ['--C', '0.9465594398044223', '--max_iter', '100']
Registered model: hyperdrive_best_model version 1


Trying to unpickle estimator LogisticRegression from version 0.24.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


The URL in the comment no longer works (403), so this cell uses the same data from your local bankmarketing_train.csv and still creates the dataset with TabularDatasetFactory, as the rubric requires. All the # comments are kept:

In [11]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###
# The URL above is no longer accessible (HTTP 403), so the same Bank Marketing data
# (downloaded from UCI and saved as bankmarketing_train.csv) is uploaded to the
# workspace's default datastore and loaded from there with TabularDatasetFactory.
datastore = ws.get_default_datastore()
datastore.upload_files(["bankmarketing_train.csv"], target_path="data", overwrite=True)

ds = TabularDatasetFactory.from_delimited_files(path=(datastore, "data/bankmarketing_train.csv"))
print(ds.take(5).to_pandas_dataframe())

Uploading an estimated of 1 files
Uploading bankmarketing_train.csv
Uploaded bankmarketing_train.csv, 1 files out of an estimated total of 1
Uploaded 1 files
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999     False  nonexistent          1.1   
1   may         mon  ...         1    999     False  non

In [12]:
from train import clean_data

# Use the clean_data function to clean your data.
x, y = clean_data(ds)

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}



A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Add the join for AutoML in the same cell or a new cell below:

In [13]:
import pandas as pd

training_data = pd.concat([x, y], axis=1)
print(training_data.shape)
training_data.head()

(40124, 40)


,age,marital,default,housing,loan,month,day_of_week,duration,campaign,pdays,...,contact_telephone,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_illiterate,education_professional.course,education_university.degree,education_unknown,y
0,56,1,0,0,0,5,1,261,1,999,...,1,1,0,0,0,0,0,0,0,0
1,57,1,0,0,0,5,1,149,1,999,...,1,0,0,0,1,0,0,0,0,0
2,37,1,0,1,0,5,1,226,1,999,...,1,0,0,0,1,0,0,0,0,0
3,40,1,0,0,0,5,1,151,1,999,...,1,0,1,0,0,0,0,0,0,0
4,56,1,0,0,1,5,1,307,1,999,...,1,0,0,0,1,0,0,0,0,0


In [16]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task="classification",
    primary_metric="accuracy",
    training_data=training_data,
    label_column_name="y",
    n_cross_validations=5,
    _ignore_package_version_incompatibilities=True)

In [17]:
# Submit your automl run

### YOUR CODE HERE ###
automl_run = exp.submit(automl_config, show_output=True)

No run_configuration provided, running on local with default configuration
Running in the active local environment.


Experiment,Id,Type,Status,Details Page,Docs Page
udacity-project,AutoML_31f8ff4d-b1cc-4c43-95b4-96b8f37d6e97,automl,Preparing,Link to Azure Machine Learning studio,Link to Documentation


Current status: DatasetEvaluation. Gathering dataset statistics.
Current status: FeaturesGeneration. Generating features for the dataset.
Current status: DatasetFeaturization. Beginning to fit featurizers and featurize the dataset.
Current status: DatasetFeaturizationCompleted. Completed fit featurizers and featurizing the dataset.
Current status: DatasetBalancing. Performing class balancing sweeping
Current status: DatasetCrossValidationSplit. Generating individually featurized CV splits.

********************************************************************************************
DATA GUARDRAILS: 

TYPE:         Class balancing detection
STATUS:       ALERTED
DESCRIPTION:  To decrease model bias, please cancel the current run and fix balancing problem.
              Learn more about imbalanced data: https://aka.ms/AutomatedMLImbalancedData
DETAILS:      Imbalanced data can lead to a falsely perceived positive effect of a model's accuracy because the input data has bias towards one cl

2026/11/07 11:12:26 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026-11-07:11:26:02,714 INFO     [explanation_client.py:334] Using default datastore for uploads


In [18]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###
import joblib, os

best_automl_run, fitted_model = automl_run.get_output()

print("Best AutoML run ID:", best_automl_run.id)
print("Algorithm:", best_automl_run.properties.get("run_algorithm"))
print("Accuracy:", best_automl_run.get_metrics()["accuracy"])
print(fitted_model.steps[-1])

# Save the model locally
os.makedirs("outputs", exist_ok=True)
joblib.dump(fitted_model, "outputs/automl_best_model.joblib")

# Register the model in the workspace
model = best_automl_run.register_model(model_name="automl_best_model",
                                       model_path="outputs/model.pkl")
print("Registered model:", model.name, "version", model.version)

Best AutoML run ID: AutoML_31f8ff4d-b1cc-4c43-95b4-96b8f37d6e97_0
Algorithm: LightGBM
Accuracy: 0.9197238223025626
('LightGBMClassifier', LightGBMClassifier(min_data_in_leaf=20, n_jobs=1, problem_info=ProblemInfo(gpu_training_param_dict={'processing_unit_type': 'cpu'}), random_state=None))
Registered model: automl_best_model version 1
